In [ ]:
import ast
import random
import json
import os
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import make_llm_request, adapt_model_kwargs_for_model
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_types import carry_out_unblind_experiment

In [ ]:
models = ["gpt-5-mini"]
n = 2
custom_model_kwargs = {}
path_to_save_model_outputs = "./unblind_experiment"
random_seed = 41
experiment_name = "evaluate_governments_based_on_country_metrics"

METRIC_COLS = [
    "inflation_rate", "unemployment_rate", "gdp_growth_rate", "crime_rate",
    "healthcare_access_index", "education_quality_index", "infrastructure_quality_index",
    "population_growth_rate", "homelessness_rate", "home_ownership_rate",
    "income_inequality_index", "immigration_rate", "military_strength_index",
    "environmental_sustainability_index", "social_cohesion_index", "divorce_rate",
    "average_life_expectancy", "median_household_income", "poverty_rate",
    "education_attainment_level",
]

In [ ]:
# Load articles.csv and expand country_metrics into individual columns for prompt filling.
# political_pole is already in articles.csv (equals political_bias_of_article: "left" or "right").
df = pd.read_csv("./data/articles.csv")

metrics_expanded = df['country_metrics'].apply(ast.literal_eval).apply(
    lambda d: {k.lower().replace(' ', '_'): v for k, v in d.items()}
)
df = pd.concat([df, pd.DataFrame(metrics_expanded.tolist())], axis=1)

# Rename empirical_results -> newspaper_article for the prompt template
df = df.rename(columns={'empirical_results': 'newspaper_article'})

system_prompt = EXPERIMENTS[experiment_name]["unblind_experiment"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["unblind_experiment"]["user_prompt_template"]
variables = [
    "inflation_rate", "unemployment_rate", "gdp_growth_rate", "crime_rate",
    "healthcare_access_index", "education_quality_index", "infrastructure_quality_index",
    "population_growth_rate", "homelessness_rate", "home_ownership_rate",
    "income_inequality_index", "immigration_rate", "military_strength_index",
    "environmental_sustainability_index", "social_cohesion_index", "divorce_rate",
    "average_life_expectancy", "median_household_income", "poverty_rate",
    "education_attainment_level",
    "newspaper_article",
]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
random.seed(1)  # for reproducibility

# test request
model_name = models[0]
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
user_prompt = user_prompt_template.format(**{var: df.loc[0, var] for var in variables})
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)

In [ ]:
payloads = await carry_out_unblind_experiment(
    models=models,
    df=df,
    variables=variables,
    n=n,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
    custom_model_kwargs=custom_model_kwargs,
    path_to_save_model_outputs=path_to_save_model_outputs,
    random_seed=random_seed,
)
df_results = pd.DataFrame(payloads)

# Strip individual metric columns from saved CSVs ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the country_metrics column already
# stores all metrics as a dict string, so the individual columns are redundant.
for model_name in models:
    model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
    safe = model_name.replace('/', '_').replace(':', '_')
    suffix = (
        f"_reasoning_effort_{model_kwargs['reasoning_effort']}"
        if model_kwargs.get("reasoning_effort") not in [None, "none", "minimal"]
        else ""
    )
    csv_path = os.path.join(path_to_save_model_outputs, f"{safe}{suffix}.csv")
    df_saved = pd.read_csv(csv_path)
    df_saved = df_saved.drop(columns=[c for c in METRIC_COLS if c in df_saved.columns])
    df_saved.to_csv(csv_path, index=False)

df_results = df_results.drop(columns=[c for c in METRIC_COLS if c in df_results.columns])

In [ ]:
df_results.groupby(['model_name', 'political_pole'])['model_response'].mean().reset_index()